In [1]:
import jax


jax.config.update("jax_debug_nans", True)
jax.config.update('jax_platform_name', 'cpu')
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import value_and_grad,grad, vmap,jit

import dynamiqs as dq

dq.set_device('cpu')

import qcsys as qs
import jaxquantum as jqt
import matplotlib.pyplot as plt
from IPython.display import clear_output
from functools import partial
import math

import optax
import warnings
warnings.filterwarnings("ignore")

### let's first optimize the transmon frequency to be fluxonium 0-7 transition

In [2]:
from flax import struct

@struct.dataclass
class MyTransmon(qs.SingleChargeTransmon):
    N_max_charge: int = struct.field(pytree_node=False)

    @classmethod
    def create(cls, N, N_max_charge, params, label=0, use_linear=False):
        return cls(N, N_max_charge, params, label, use_linear, N_max_charge)
    
    def get_H_full(self):
        #  consistant with scqubits 
        dimension = 2 * self.N_max_charge + 1
        def generate_hamiltonian_element(ind, Ec, N_max_charge, ng):
            return 4.0 * Ec * (ind - N_max_charge - ng) ** 2

        dim_range = jnp.arange(dimension)
        hamiltonian_mat = jnp.diag(vmap(generate_hamiltonian_element, in_axes=(0, None, None, None))(dim_range, self.params["Ec"], self.N_max_charge, self.params["ng"]))
        ind = jnp.arange(dimension - 1)
        hamiltonian_mat = hamiltonian_mat.at[ind, ind + 1].set(-self.params["Ej"] / 2.0)
        hamiltonian_mat = hamiltonian_mat.at[ind + 1, ind].set(-self.params["Ej"] / 2.0)

        return jqt.Qarray.create(hamiltonian_mat)
    def build_n_op(self):
        return jqt.Qarray.create(jnp.diag(jnp.arange(-self.N_max_charge, self.N_max_charge + 1)))
    @jit
    def get_op_in_H_eigenbasis(self, op):
        if type(op) == jqt.Qarray:
            op = op.data
        evecs = self.eig_systems["vecs"][:, : self.N]
        op = jnp.dot(jnp.conjugate(evecs.transpose()), jnp.dot(op, evecs))
        return jqt.Qarray.create(op)

In [3]:
# Ec_t = 0.2
# def objective(params):
#     Ej_t = params[0]
#     qst = MyTransmon.create(
#         N = 10,
#         params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
#         N_max_charge=30
#     )
#     return abs(qst.eig_systems["vals"][1] - qst.eig_systems["vals"][0] - 7.182920828753576)


# optimizer = optax.adam(learning_rate=0.1) 
# params = jnp.array([40.0])
# opt_state = optimizer.init(params)

# num_steps = 1000
# for step in range(num_steps):
#     val, grads = jax.value_and_grad(objective)(params)
#     if step %10 == 0:
#         clear_output()
#         print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
#     if val < 1e-6:
#         clear_output()
#         print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
#         break
#     updates, opt_state = optimizer.update(grads, opt_state)
#     params = optax.apply_updates(params, updates)

# print(f'Optimized params: {params}')

# Now let's define the relative functions for our objective

In [4]:
def calculate_eig(Ns, H: jqt.Qarray):
    N_tot = math.prod(Ns)
    vals, kets = jnp.linalg.eigh(H.data)

    ketsT = kets.T

    def get_product_idx(edx):
        argmax = jnp.argmax(jnp.abs(ketsT[edx]))
        return  argmax  # product index
    edxs = jnp.arange(N_tot)
    product_indices_sorted_by_eval = vmap(get_product_idx)(edxs)
    return (vals,kets,product_indices_sorted_by_eval) # Here kets is equivalent to the S in qutip.Qobj.transform

def find_closest_dressed_index(product_index, product_indices_sorted_by_eval):
    dressed_index = jnp.argmin(jnp.abs(product_index - product_indices_sorted_by_eval))
    return dressed_index.item()

def transform_op_into_dressed_basis_jax(op_matrix: jqt.Qarray, 
                                        S: jax.Array) -> jax.Array:
    """
    Transform an operator into the dressed basis using JAX.

    Parameters:
    - op_matrix: A 2D JAX array representing the operator's matrix.
    - S: A 2D JAX array representing the dressed eigenvectors similar to the S in qutip.Qobj.transform

    Returns:
    - A 2D JAX array representing the transformed operator.
    """
    data = jnp.dot(S, jnp.dot(op_matrix.data, S.T.conj()))
    return data

def square_pulse_with_rise_fall(t,
                                args = {}):
    
    w_d = args['w_d']
    amp = args['amp']
    t_start = args.get('t_start', 0)  # Default start time is 0
    t_rise = args.get('t_rise', 0)  # Default rise time is 0 for no rise
    t_square = args.get('t_square', 0)  # Duration of constant amplitude

    def cos_modulation():
        return 2 * jnp.pi * amp * jnp.cos(w_d * 2 * jnp.pi * t)
    
    t_fall_start = t_start + t_rise + t_square  # Start of fall
    t_end = t_fall_start + t_rise  # End of the pulse

    before_pulse_start = jnp.less(t, t_start)
    during_rise_segment = jnp.logical_and(jnp.greater(t_rise, 0), jnp.logical_and(jnp.greater_equal(t, t_start), jnp.less_equal(t, t_start + t_rise)))
    constant_amplitude_segment = jnp.logical_and(jnp.greater(t, t_start + t_rise), jnp.less_equal(t, t_fall_start))
    during_fall_segment = jnp.logical_and(jnp.greater(t_rise, 0), jnp.logical_and(jnp.greater(t, t_fall_start), jnp.less_equal(t, t_end)))

    return jnp.where(before_pulse_start, 0,
                    jnp.where(during_rise_segment, jnp.sin(jnp.pi * (t - t_start) / (2 * t_rise)) ** 2 * cos_modulation(),
                            jnp.where(constant_amplitude_segment, cos_modulation(),
                                        jnp.where(during_fall_segment, jnp.sin(jnp.pi * (t_end - t) / (2 * t_rise)) ** 2 * cos_modulation(), 0))))


In [5]:
solver = dq.solver.Tsit5(
                    rtol= 1e-06,
                    atol= 1e-06,
                    safety_factor= 0.9,
                    min_factor= 0.2,
                    max_factor = 5.0,
                    max_steps = int(1e4*1000),
                )

n_lvls_fluxonium = 20
n_lvls_transmon = 4

tot_dim = n_lvls_fluxonium*n_lvls_transmon

Ej_f = 2.7
Ec_f = 0.6
El_f = 0.13
t_rise = 3.3

truncated_dim = 26 # will include 7,1
def truncate(data: jnp.array):
    return data[:truncated_dim,:truncated_dim]

In [14]:

def objective(params):

    Ej_t = 34.12111946
    pulse_length = params[0]
    amp_with_2pi = params[1]
    g_tf = 0.2

    t_tot = t_rise + pulse_length
    # tlist = jnp.linspace(0,t_tot,jnp.array(t_tot,int))
    
    tlist = jnp.linspace(0,t_tot,jnp.array(t_tot,int))

    qsf = qs.Fluxonium.create(
        n_lvls_fluxonium,
        {"Ej": Ej_f, "Ec": Ec_f, "El": El_f, "phi_ext": 0.0},
        N_pre_diag=100,
        use_linear = False
        )
    Ec_t = 0.2
    qst = MyTransmon.create(
        N = n_lvls_transmon,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=30
        )
    devices = [qsf, qst]
    f_indx = 0
    t_indx = 1
    Ns = [device.N for device in devices]
    tn = qs.promote(qst.ops['n'], t_indx, Ns)
    fn = qs.promote(qsf.ops["n"], f_indx, Ns)
    system = qs.System.create(devices, couplings=[
        g_tf *  fn @ tn
        ])
    system.params["g_tf"] = g_tf
    system_evals_in_product_indices, system_evecs_in_product_indices = system.calculate_eig_linear()
    system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
    driven_op = transform_op_into_dressed_basis_jax(tn, system_evecs_sorted.T)

    w_d = system_evals_in_product_indices[0,1] - system_evals_in_product_indices[0,0]
    pulse_shape_args={
        'w_d': w_d,
        'amp': amp_with_2pi/(2*jnp.pi),
        't_rise': t_rise,
        't_square': pulse_length - t_rise
    }      

    def _H(t):
        _H =  2 * jnp.pi *truncate(jnp.diag(system_evals_sorted))
        _H += truncate(driven_op) * square_pulse_with_rise_fall(t, pulse_shape_args)
        return _H 
    H =  dq.timecallable(_H)
    
    psi0_list = [truncate(dq.basis(tot_dim, find_closest_dressed_index(fl*qst.N + 0, product_indices_sorted_by_eval)))
                        for fl in range(3)]
    result = dq.sesolve(
                H = H,
                psi0 = psi0_list,
                tsave = tlist,
                solver = solver
                )
    
    f0_e = dq.expect(
                truncate(dq.basis_dm(tot_dim, find_closest_dressed_index(0*qst.N + 1, product_indices_sorted_by_eval))),
                result.states[0][-1]
                ).real
    f1_e = dq.expect(
                truncate(dq.basis_dm(tot_dim, find_closest_dressed_index(1*qst.N + 1, product_indices_sorted_by_eval))),
                result.states[1][-1]
                ).real
    f2_e = dq.expect(
                truncate(dq.basis_dm(tot_dim, find_closest_dressed_index(2*qst.N + 1, product_indices_sorted_by_eval))),
                result.states[2][-1]
                ).real
    
    return 1 - f0_e + f1_e + f2_e # we minimize the objective

In [15]:
params =  jnp.array([
                     250.0,
                     0.011026707187734986
                    ])

optimizer = optax.adam(learning_rate=0.1) 
opt_state = optimizer.init(params)

num_steps = 1000
func = jit(jax.value_and_grad(objective))
for step in range(num_steps):
    val, grads = func(params)
    clear_output()
    print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
    if val < 1e-4:
        break
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

ConcretizationTypeError: Abstract tracer value encountered where concrete value is expected: traced array with shape int64[].
This occurred in the item() method of jax.Array
The error occurred while tracing the function objective at /tmp/ipykernel_158009/4126359320.py:1 for jit. This value became a tracer due to JAX operations on these lines:

  operation a:f64[400] = pjit[
  name=_linspace
  jaxpr={ lambda ; b:i64[] c:i64[]. let
      d:f64[] = convert_element_type[new_dtype=float64 weak_type=False] b
      e:f64[] = convert_element_type[new_dtype=float64 weak_type=False] c
      f:f64[] = sub e d
      _:f64[] = div f 399.0
      g:f64[399] = iota[dimension=0 dtype=float64 shape=(399,)] 
      h:f64[399] = div g 399.0
      i:f64[1] = reshape[dimensions=None new_sizes=(1,)] d
      j:f64[399] = sub 1.0 h
      k:f64[399] = mul i j
      l:f64[1] = reshape[dimensions=None new_sizes=(1,)] e
      m:f64[399] = mul l h
      n:f64[399] = add k m
      o:f64[1] = broadcast_in_dim[broadcas

In [ ]:
params =  jnp.array([
                     3.0,
                     0.011026707187734986
                     ])


grad(objective)(params)

# objective(params)

Array(1.0006945, dtype=float64)

In [9]:
qst = MyTransmon.create(
        N = n_lvls_transmon,
        params = {"Ej": 34.0, "Ec": 0.2,"ng":0.0},
        N_max_charge=30
        )

qst.get_H_full()

Quantum array: dims = [[61], [61]], shape = (61, 61), type = oper
Qarray data =
[[720. +0.j -17. +0.j   0. +0.j ...   0. +0.j   0. +0.j   0. +0.j]
 [-17. +0.j 672.8+0.j -17. +0.j ...   0. +0.j   0. +0.j   0. +0.j]
 [  0. +0.j -17. +0.j 627.2+0.j ...   0. +0.j   0. +0.j   0. +0.j]
 ...
 [  0. +0.j   0. +0.j   0. +0.j ... 627.2+0.j -17. +0.j   0. +0.j]
 [  0. +0.j   0. +0.j   0. +0.j ... -17. +0.j 672.8+0.j -17. +0.j]
 [  0. +0.j   0. +0.j   0. +0.j ...   0. +0.j -17. +0.j 720. +0.j]]

In [ ]:


def get_esys(EJ):
    qst = MyTransmon.create(
            N = n_lvls_transmon,
            params = {"Ej": EJ, "Ec": 0.2,"ng":0.0},
            N_max_charge=30
            )
    return qst._calculate_eig_systems()
jax.make_jaxpr(jax.grad(get_esys)(3.0))

FloatingPointError: invalid value (nan) encountered in jit(integer_pow). Because jax_config.debug_nans.value and/or config.jax_debug_infs is set, the de-optimized function (i.e., the function as if the `jit` decorator were removed) was called in an attempt to get a more precise error message. However, the de-optimized function did not produce invalid values during its execution. This behavior can result from `jit` optimizations causing the invalid value to be produced. It may also arise from having nan/inf constants as outputs, like `jax.jit(lambda ...: jax.numpy.nan)(...)`. 

It may be possible to avoid the invalid value by removing the `jit` decorator, at the cost of losing optimizations. 

If you see this error, consider opening a bug report at https://github.com/google/jax.